# Создание веб-приложения для LLM: примеры на Hugging Face Transformers, Gradio и Streamlit

**В этом ноутбуке**:

* Детально рассмотрим, как создать простое веб-приложение для взаимодействия с
большой языковой моделью (LLM) с помощью библиотек **Hugging Face Transformers**, **Gradio** и **Streamlit**.
* В качестве примера будем использовать инструкционно-тренированную модель `Qwen/Qwen2.5-0.5B-Instruct` от Hugging Face в локальном режиме.
* Реализуем возможность переключения на альтернативный режим работы через GigaChat API, используя ключ API и выбранную модель. Все приложение будет настраиваться через аргументы командной строки или переменные окружения, что позволит гибко выбирать модель и режим работы.

**Краткий обзор структуры ноутбука:**

* Установка необходимых библиотек и зависимостей.
* Загрузка и инициализация LLM-модели Hugging Face (`Qwen2.5` по умолчанию).
* Обработка пользовательского ввода и генерация ответа локально.
* Альтернативный режим через GigaChat API (с использованием ключа и модели).
* Пример настройки через аргументы командной строки и переменные окружения.
* Создание веб-интерфейса с помощью Gradio.
* Создание веб-интерфейса с помощью Streamlit.
* Пример запуска и использования приложения.

Каждый раздел будет сопровождаться подробными пояснениями на русском языке, демонстрационным кодом Python и комментариями. Рассмотрим каждую часть по порядку.

## Установка зависимостей

Прежде всего для работы нам понадобятся следующие Python-библиотеки:
* `transformers` — для загрузки и использования моделей Hugging Face.
* `gradio` — для создания простого веб-интерфейса.
* `streamlit` — для создания веб-приложения.
* `gigachat` — для работы с GigaChat API (если планируете альтернативный режим).

Выполните установку (в терминале или в ноутбуке) командой:

In [1]:
%pip install transformers==4.52.4 gradio==5.31.0 streamlit==1.46.0 gigachat -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 55.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.2/54.2 MB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 80.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.1/323.1 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 444.8/444.8 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 76.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 73.9 MB/s eta 0:00:00


## Загрузка и инициализация модели Hugging Face (локально)

Мы будем использовать модель `Qwen/Qwen2.5-0.5B-Instruct` по умолчанию, но сделаем так, чтобы модель можно было задавать через параметры приложения. Для загрузки модели применим классы `AutoTokenizer` и `AutoModelForCausalLM` из `transformers`.

Вот пример кода для загрузки модели и токенизатора (мы используем рекомендуемый Hugging Face подход с `device_map="auto"`, что позволяет автоматически распределить модель по доступным устройствам (CPU/GPU)):

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Задаем название модели (по умолчанию Qwen2.5 0.5B Instruct)
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

# Загружаем модель и токенизатор (при больших моделях может потребоваться много памяти)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",   # автоматически подбираем тип (float16/float32) в зависимости от оборудования
    device_map="auto"     # автоматическое распределение на CPU/GPU
)
tokenizer = AutoTokenizer.from_pretrained(model_name)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

## Обработка пользовательского ввода и генерация ответа (локальный режим)

Предположим, у нас есть пользовательский ввод (prompt) и мы хотим сгенерировать ответ с помощью нашей модели. Для этого пишем функцию, которая принимает строку запроса и возвращает сгенерированный текст. Пример для модели `Qwen`, которая хорошо поддерживает чат-ориентированный ввод:

In [3]:
def generate_with_local(model, tokenizer, user_input):
    """
    Генерирует ответ локальной моделью Hugging Face.
    """
    # Формируем сообщения в стиле чат-модели (system + user)
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": user_input}
    ]
    # Создаем единую строку запроса с помощью шаблона токенизатора
    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # Токенизируем запрос и переносим тензоры на устройство модели
    inputs = tokenizer([prompt_text], return_tensors="pt").to(model.device)

    # Генерируем ответ
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,       # ограничиваем число генерируемых токенов
        do_sample=False           # делаем жадную генерацию (без случайности)
    )

    # Извлекаем только сгенерированную часть (после входных токенов)
    gen_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    # Декодируем токены в строку
    response = tokenizer.decode(gen_tokens, skip_special_tokens=True)
    return response

# Пример использования функции
user_input = "Привет, как дела?"
response = generate_with_local(model, tokenizer, user_input)
print("Пользователь:", user_input)
print("Модель ответила:", response)


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Пользователь: Привет, как дела?
Модель ответила: Хорошо! Я здесь, чтобы помочь вам. Как я могу помочь вам сегодня?


Обратите внимание:

* Мы добавляем системное сообщение (`"You are a helpful assistant."`) — это часть контекста, которая помогает модели понять свою роль.
* Функция `apply_chat_template` (специальная для `Qwen` и некоторых других chat-моделей) объединяет сообщения в один prompt с учетом специальных шаблонов.
* Параметр `max_new_tokens` ограничивает длину ответа. Его можно регулировать в зависимости от задачи.
* Здесь мы не используем семплирование (`do_sample=False`), чтобы получить детерминированный ответ. Можно включить случайность при желании.


## Альтернативный режим через GigaChat API

Вместо локальной модели можно и нужно использовать GigaChat API (например, модели `GigaChat-2` или другие). Для этого нужно установить библиотеку `gigachat` и иметь действующий API-ключ.

### Инициализация GigaChat API

Получим ключ из переменной окружения `GIGACHAT_API_KEY`. Если он задан, переключимся в режим GigaChat:

In [22]:
import os
import getpass

os.environ['GIGACHAT_API_KEY'] = getpass.getpass(prompt='GIGACHAT_API_KEY: ', stream=None)

GIGACHAT_API_KEY: ··········


In [23]:
import os
from rich import print
from gigachat import GigaChat
from gigachat.models import Chat, Messages, MessagesRole

# Получаем ключ из переменной окружения
giga_api_key = os.getenv("GIGACHAT_API_KEY", "")

# Создаем клиента GigaChat
client = None
use_gigachat = False

if giga_api_key:
    client = GigaChat(
        credentials=giga_api_key,
        scope="GIGACHAT_API_B2B",
        model="GigaChat-2-Max",
        verify_ssl_certs=False,
        profanity_check=False,
        timeout=120
    )
    use_gigachat = True
    print("Режим GigaChat: включён")
else:
    print("Режим GigaChat: отключён, используется локальная модель")

Режим GigaChat: включён

Если вы запустите приложение командой с предварительно экспортированной переменной `GIGACHAT_API_KEY`, то режим GigaChat автоматически включится.

In [24]:
# Получить список моделей

response = client.get_models()

print(response)

Models(
    x_headers={
        'x-request-id': 'd1f5b224-7d9b-4448-b564-bd78cc036d8b',
        'x-session-id': 'f54c670f-201f-4e0d-9474-a51ddbe6d65f',
        'x-client-id': None
    },
    data=[
        Model(x_headers=None, id_='GigaChat', object_='model', owned_by='salutedevices'),
        Model(x_headers=None, id_='GigaChat-2', object_='model', owned_by='salutedevices'),
        Model(x_headers=None, id_='GigaChat-2-Max', object_='model', owned_by='salutedevices'),
        Model(x_headers=None, id_='GigaChat-2-Pro', object_='model', owned_by='salutedevices'),
        Model(x_headers=None, id_='GigaChat-Max', object_='model', owned_by='salutedevices'),
        Model(x_headers=None, id_='GigaChat-Max-preview', object_='model', owned_by='salutedevices'),
        Model(x_headers=None, id_='GigaChat-Plus', object_='model', owned_by='salutedevices'),
        Model(x_headers=None, id_='GigaChat-Pro', object_='model', owned_by='salutedevices'),
        Model(x_headers=None, id_='GigaChat-Pro-preview', object_='model', owned_by='salutedevices'),
        Model(x_headers=None, id_='GigaChat-preview', object_='model', owned_by='salutedevices'),
        Model(x_headers=None, id_='Embeddings', object_='model', owned_by='salutedevices'),
        Model(x_headers=None, id_='Embeddings-2', object_='model', owned_by='salutedevices'),
        Model(x_headers=None, id_='EmbeddingsGigaR', object_='model', owned_by='salutedevices'),
        Model(x_headers=None, id_='GigaEmbeddings-3B-2025-09', object_='model', owned_by='salutedevices')
    ],
    object_='list'
)

## Генерация через GigaChat API

Теперь опишем функцию, которая отправляет запрос к GigaChat. Будем использовать чат-метод `chat` с моделью (по умолчанию, например, `GigaChat-2`).

In [25]:
def generate_with_gigachat(prompt, model_name="GigaChat-2-Max"):
    """
    Генерирует ответ через GigaChat API.
    """

    # Формируем сообщения
    messages = [
        Messages(
            role=MessagesRole.SYSTEM,
            content="Ты очень полезный помощник. Ты всегда отвечаешь в стихах."
        ),
        Messages(
            role=MessagesRole.USER,
            content=prompt
        )
    ]

    # Отправляем запрос к GigaChat
    with GigaChat(credentials=giga_api_key, scope="GIGACHAT_API_B2B", verify_ssl_certs=False, model=model_name) as giga:
        response = giga.chat(Chat(messages=messages))

    # Извлекаем текст ответа
    return response.choices[0].message.content.strip()


# Пример использования функции (если ключ установлен)
if use_gigachat:
    prompt = "Расскажи шутку о компьютерах."
    answer = generate_with_gigachat(prompt, model_name="GigaChat-2-Max")
    print("Prompt:", prompt)
    print("Answer:", answer)

Prompt: Расскажи шутку о компьютерах.

Answer: Системный блок решил, что пора худеть,
Отключился от сети и стал на диету.
Но процессор ему сказал: "Друг мой, не мучай себя!
Лучше поставь обновление — сразу ускоришься!"

Если `GIGACHAT_API_KEY` не установлен, этот код не будет запущен (`use_gigachat=False`). В режиме GigaChat мы не используем локальную модель `Qwen` вовсе.

## Настройка аргументов командной строки и переменных окружения

Обычно приложение запускают из командной строки, передавая параметры. Рассмотрим пример использования `argparse` для выбора модели и режима работы:

In [17]:
import argparse

def parse_args():
    parser = argparse.ArgumentParser(description="LLM Web App")
    parser.add_argument("--model", type=str, default=None, help="Hugging Face model name")
    parser.add_argument("--use_gigachat", action="store_true", help="Use GigaChat API instead of local model")
    return parser.parse_args()

# Парсим аргументы (пример использования в скрипте, в Jupyter не применяется)
# args = parse_args()
# model_name = args.model or model_name  # если указано --model, перезаписываем
# use_gigachat = args.use_gigachat
# gigachat_model = args.gigachat_model


Внутри Jupyter мы не будем вызывать `parse_args()` (это подходит для скрипта `app.py`). Но идея в том, что пользователь может запускать программу так:

```python
python app.py --model "EleutherAI/gpt-j-6B" --use_gigachat
```

* Параметр `--model` задает название Hugging Face-модели (если `--use_gigachat` не указан, будет загружена эта модель).
* Флаг `--use_gigachat` говорит, что нужно игнорировать локальную модель и использовать GigaChat API (тогда `--gigachat_model` можно указать название модели GigaChat).
* Также всегда можно задавать переменные окружения `MODEL_NAME`, `GIGACHAT_API_KEY` и т.п.


Например, можно запустить так:
```python
export MODEL_NAME="google/flan-t5-small"
export GIGACHAT_API_KEY="ваш_ключ"
python app.py --use_gigachat
```

В этом случае по умолчанию приложение активирует GigaChat (из-за ключа) и сгенерирует ответ с помощью указанной `--gigachat_model` (по умолчанию `"GigaChat-2"`). Если флаг `--use_gigachat` не указан, приложение будет использовать локальную модель `google/flan-t5-small`. В итоге получаем гибкую настройку: модель и режим задаются либо переменными окружения, либо аргументами командной строки.

## Создание веб-интерфейса с Gradio

**Gradio** позволяет быстро создать простой веб-интерфейс для функции генерации текста. Пойдем по плану: создадим функцию `chat_interface`, которая принимает текстовый запрос и возвращает ответ, выбирая между локальной моделью и GigaChat. Затем передадим ее в `gr.Interface`.

## Пример кода на Gradio

In [18]:
import gradio as gr

# Определяем функцию, которую будет вызывать интерфейс
def chat_interface(user_input):
    if use_gigachat:
        # Если включен режим GigaChat, генерируем через GigaChat API
        # return generate_with_gigachat(user_input, model_name=args.gigachat_model if args.use_gigachat else "GigaChat-2")
        return generate_with_gigachat(user_input, model_name="GigaChat-2-Max")
    else:
        # Иначе используем локальную модель Hugging Face
        return generate_with_local(model, tokenizer, user_input)

# Создаем интерфейс Gradio
iface = gr.Interface(
    fn=chat_interface,
    inputs=gr.Textbox(
        lines=4,
        placeholder="Введите ваш запрос здесь...",
        label="Входное сообщение",
        show_copy_button=True
    ),
    outputs=gr.Textbox(
        lines=12,
        label="Ответ модели",
        show_copy_button=True,
        show_label=True,
        max_lines=20
    ),
    title="LLM Chat Application",
    description="🚀 Введите ваш запрос в поле ниже и получите ответ от языковой модели",
    theme="soft",
    allow_flagging="never"
)
iface.launch()  # Для запуска Gradio

/usr/local/lib/python3.12/dist-packages/gradio/interface.py:416: UserWarning: The `allow_flagging` parameter in `Interface` is deprecated.Use `flagging_mode` instead.
  warnings.warn(


It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://694124c4948889ed86.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Пояснения:

* Функция `chat_interface` определяет единый интерфейс: если `use_gigachat=True`, вызывается `generate_with_gigachat`, иначе — `generate_with_local`. Мы используем наши ранее написанные функции.
* `gr.Interface` создает одностраничный веб-интерфейс с заголовком, полем для ввода и полем для вывода текста.
* При запуске `iface.launch()` Gradio стартует локальный сервер и выдаст URL, по которому можно открыть приложение в браузере (например, `http://127.0.0.1:7860`).
* Параметры `title` и `description` задают заголовок и описание страницы.

**Важно**: Gradio в Jupyter-ноутбуке попытается открыть интерфейс внутри самого ноутбука или в браузере. В сценариях обучения удобно просмотреть его по ссылке. Чтобы не запускать каждую ячейку (что неудобно в демонстрационных целях), мы можем просто описать, как это делается, и показать код.

## Создание веб-интерфейса со Streamlit

**Streamlit** — еще одна популярная библиотека для создания простых веб-приложений. Напишем скрипт, который использует Streamlit для общения с LLM. Примерный код может выглядеть так:

In [19]:
# Импортируем Streamlit
import streamlit as st

st.set_page_config(page_title="LLM Chat App", layout="centered")
st.title("LLM Chat App")
st.write("Это простое приложение для общения с LLM через Hugging Face или GigaChat.")

# Поле для ввода запроса пользователем
user_input = st.text_area("Введите ваш запрос:", height=100)

# Кнопка для генерации ответа
if st.button("Сгенерировать ответ"):
    if not user_input:
        st.warning("Пожалуйста, введите запрос.")
    else:
        if use_gigachat:
            answer = generate_with_gigachat(user_input, model_name=args.gigachat_model if args.use_gigachat else "GigaChat-2")
        else:
            answer = generate_with_local(model, tokenizer, user_input)
        st.subheader("Ответ модели:")
        st.write(answer)


2026-01-31 16:25:38.441 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-31 16:25:38.442 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-31 16:25:38.443 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-31 16:25:38.444 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-31 16:25:38.446 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-31 16:25:38.449 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-31 16:25:38.449 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-31 16:25:38.450 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

## Пояснения:

* Мы используем `st.title`, `st.write` и `st.text_area` для создания элементов интерфейса.
* `st.button("Сгенерировать ответ")` — кнопка, по нажатию которой выполняется генерация.
* После генерации результат выводится через `st.subheader` и `st.write`.
* Переменная `user_input` хранит текст запроса, введенный пользователем.
* Переключение между режимами GigaChat и локальной моделью реализуем так же, как и в Gradio.

Чтобы запустить это приложение, необходимо сохранить код в Python-файл (например, `app.py`) и выполнить в терминале:

```bash
streamlit run app.py -- --model "Qwen/Qwen2.5-0.5B-Instruct"
```

**Обратите внимание**: если мы используем `streamlit run`, то аргументы после `--` передаются нашему скрипту (это особенность Streamlit). В примере мы передаем аргумент `--model` со значением (если он есть) уже после `--`. Также можно задать `--use_gigachat` аналогичным образом:

```bash
streamlit run app.py -- --use_gigachat --gigachat_model GigaChat-2
```

Streamlit запустит приложение в браузере (обычно по адресу `http://localhost:8501`). В нем будет заголовок, поле для ввода и кнопка, а после нажатия вы увидите ответ модели.

В этом примере мы показали сразу и Gradio, и Streamlit в одном файле. На практике вы, скорее всего, будете использовать либо Gradio (для быстрого прототипа), либо Streamlit (для более масштабируемого приложения). Функции генерации (`generate_with_local` и `generate_with_gigachat`) одинаково подходят под оба интерфейса.

## Запуск и проверка работы приложения


**Gradio**: если вы используете Gradio (как в примере выше), достаточно вызвать `iface.launch()`. В ноутбуке при этом появится ссылка. Если код перенести в файл `app.py`, то можно запускать из терминала:

In [20]:
! python app.py --model "Qwen/Qwen2.5-0.5B-Instruct"

python3: can't open file '/content/app.py': [Errno 2] No such file or directory


**Streamlit**: если вы предпочитаете Streamlit, сохраните код в файл (например, `app.py`) и выполните:


```bash
streamlit run app.py -- --model "Qwen/Qwen2.5-0.5B-Instruct"
```

Streamlit запустится и тоже выдаст локальный адрес. Интерфейс Streamlit отличается, но суть та же: есть поле для ввода и кнопка для генерации.

В обоих случаях проверьте работу: введите произвольный запрос и убедитесь, что модель отвечает. Вы можете переключаться между локальной моделью и GigaChat, задавая соответствующие флаги и переменные окружения.

## Заключение

Мы рассмотрели полный цикл создания простого веб-приложения для взаимодействия с LLM-моделью:
* Узнали, как загружать и инициализировать модель Hugging Face Transformer (`Qwen/Qwen2.5-0.5B-Instruct` по умолчанию) в локальном режиме.
* Написали функции генерации ответа как локально (с использованием токенизации и `model.generate`), так и через GigaChat API (с использованием ключа и чат-интерфейса).
* Настроили приложение так, чтобы можно было выбирать модель и режим работы через аргументы командной строки или переменные окружения (`MODEL_NAME`, `GIGACHAT_API_KEY`, флаги `--use_gigachat` и т.п.).
* Создали два варианта веб-интерфейса: **Gradio** и **Streamlit**, продемонстрировали, как быстро собрать UI для общения с моделью.
* Включили развернутые пояснения, примеры кода и объяснили ключевые моменты.

Такой подход позволяет легко масштабировать и адаптировать приложение: можно менять модель, добавлять дополнительные настройки, улучшать интерфейс. Gradio подходит для быстрой демонстрации и прототипов, Streamlit — для более сложных веб-приложений. В обоих случаях код генерации остается общим.

Вы можете поэкспериментировать с разными моделями Hugging Face (загрузив другие имена через `--model`) или подключить новые возможности GigaChat API (например, streaming, разные модели GigaChat-2-Max и т.д.). Главное — инкапсулировать логику генерации в функции и использовать гибкие параметры запуска, как мы сделали.

Желаем успехов в разработке и дальнейшем изучении LLM-веб-приложений!